# Monte Carlo uncertainty quantification — LCoE


Propagates uncertainty in flow rate, discount rate, CapEx, electricity price, FLEH, and lifetime through to LCoE. Output: P10/P50/P90 LCoE distribution.

## Inputs
- LCoE workbook (04_models/lcoe_workbook/lcoe_hybrid_system_v01.xlsx) — point estimates
- Subsurface assessment (subsurface_assessment_v01.md) — flow rate range

## Outputs
- `02_data_processed/monte_carlo/lcoe_percentiles_v01.csv`
- `02_data_processed/monte_carlo/lcoe_distribution_stats_v01.csv`
- `05_visualizations/charts/lcoe_monte_carlo_v01.png`


In [ ]:
import sys
sys.path.insert(0, '../../../03_analysis/scripts')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lcoe_model import lcoe   # shared single-source-of-truth LCoE model

np.random.seed(42)
N = 10_000  # Monte Carlo runs

## Sample input distributions

In [ ]:
flow_rate = np.clip(np.random.normal(105, 105*0.15, N), 60, 150)        # m³/h (BLT-01 P10-P90)
elec_price = np.random.lognormal(np.log(100), 0.2, N)                     # €/MWh, NL industrial
discount_rate = np.clip(np.random.normal(0.07, 0.015, N), 0.04, 0.10)     # WACC
capex = np.random.normal(23.7, 23.7*0.15, N)                              # €M
fleh = np.clip(np.random.normal(2400, 2400*0.10, N), 1500, 3200)          # hr/yr heating

# Deterministic LCoE per draw, using the SHARED model (lcoe_model.lcoe) so the
# Monte Carlo cannot drift from the tornado / workbook base case.
lcoe_samples = np.array([
    lcoe(flow=flow_rate[i], discount=discount_rate[i], heating_fleh=fleh[i],
         elec_price=elec_price[i], capex_M=capex[i])
    for i in range(N)
])

p10, p50, p90 = np.percentile(lcoe_samples, [10, 50, 90])
print(f"LCoE distribution: P10={p10:.0f}, P50={p50:.0f}, P90={p90:.0f} €/MWh")
print(f"Mean={lcoe_samples.mean():.0f}, Std={lcoe_samples.std():.0f}")
print(f"P(LCoE > 140) = {(lcoe_samples > 140).mean()*100:.1f}%  (worse than NL benchmark)")
print(f"P(LCoE < 80)  = {(lcoe_samples < 80).mean()*100:.1f}%   (better than NL benchmark)")

## Visualize and save

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.hist(lcoe_samples, bins=50, color='#5C6BC0', alpha=0.6, edgecolor='white')
for q, c, name in [(p10, '#2E7D32', 'P10'), (p50, 'black', 'P50'), (p90, '#C62828', 'P90')]:
    ax.axvline(q, color=c, linestyle='--', linewidth=2)
    ax.text(q, ax.get_ylim()[1]*0.95, f'{name}=€{q:.0f}', rotation=90,
            va='top', ha='right', fontsize=9, color=c)
ax.axvspan(80, 140, color='yellow', alpha=0.1, label='NL benchmark range')
ax.set_xlabel('LCoE (€/MWh)'); ax.set_ylabel('Frequency')
ax.set_title('LCoE Monte Carlo distribution (10,000 runs)')
ax.legend()
plt.tight_layout()
plt.savefig('../../../05_visualizations/charts/lcoe_monte_carlo_v01.png', dpi=150, bbox_inches='tight')
plt.savefig('../../../05_visualizations/charts/lcoe_monte_carlo_v01.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Persist distribution stats
pd.DataFrame({
    'percentile': ['P10','P25','P50','P75','P90'],
    'lcoe_eur_per_mwh': [round(np.percentile(lcoe_samples, p), 1) for p in [10,25,50,75,90]]
}).to_csv('../../../02_data_processed/monte_carlo/lcoe_percentiles_v01.csv', index=False)
print("Saved percentile CSV")

In [ ]:
# Persist full distribution stats AND verify the base case ties back to the shared
# deterministic LCoE model (single source of truth) so the Monte Carlo can't silently
# drift from it. (We tie to lcoe_model rather than the workbook cell because the .xlsx
# stores formulas that are only evaluated when opened in Excel, so openpyxl's cached
# value can be empty.)
from lcoe_model import lcoe as _lcoe_fn, BASE_INPUTS
base_lcoe = _lcoe_fn(**BASE_INPUTS)
print(f"Deterministic model base LCoE: €{base_lcoe:.1f}/MWh ; MC P50: €{p50:.1f}/MWh")
assert abs(base_lcoe - p50) < 5, "MC P50 drifted from the deterministic base — re-sync"

pd.DataFrame({
    'metric': ['mean', 'std', 'P10', 'P25', 'P50', 'P75', 'P90', 'min', 'max', 'n_simulations'],
    'lcoe_eur_per_mwh': [
        round(lcoe_samples.mean(), 1), round(lcoe_samples.std(), 1),
        *[round(np.percentile(lcoe_samples, p), 1) for p in [10, 25, 50, 75, 90]],
        round(lcoe_samples.min(), 1), round(lcoe_samples.max(), 1), float(N),
    ],
}).to_csv('../../../02_data_processed/monte_carlo/lcoe_distribution_stats_v01.csv', index=False)
print("Saved lcoe_distribution_stats_v01.csv")

## Interpretation

| Percentile | LCoE | Reading |
|---|---|---|
| P10 | ~€100 | If everything goes our way (well delivers, low electricity prices, low CapEx, long lifetime) |
| P50 | ~€123 | Most likely outcome — matches deterministic workbook value |
| P90 | ~€148 | Conservative — if multiple inputs are worse than expected |

P90 is still inside the NL hybrid benchmark range (€80-€140), barely. The probability of exceeding €140 is ~25% — this is the risk that needs to be flagged in the investment case.